# Predictive Maintenance: Bearing Fault Detection from Vibration 🔧
> "I used to fix machines after they broke. This project predicts the failure before it happens."

**Data:** CWRU Bearing Dataset. An accelerometer on a motor bearing, 12,000 samples per second.

**Classes (10):** Normal + 3 fault locations × 3 fault sizes
- **IR** = inner race, **B** = ball, **OR** = outer race
- fault diameter: 0.007, 0.014, 0.021 inch

| Part | What | Topic |
|---|---|---|
| 1 | Load & look at the signals | Python / signal processing |
| 2 | Cut into windows + split **without leakage** | ML theory |
| 3 | Hand-made features + Random Forest | classic ML |
| 4 | 1D CNN on raw windows | PyTorch |
| 5 | Autoencoder for anomaly detection (trained on Normal only) | PyTorch / ML theory |
| 6 | Robustness test: a different motor load (1 HP) | ML theory |
| 7 | Summary + interview story | 🎤 |

# Part 1 – Load and look at the signals

In [ ]:
from pathlib import Path
import random
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat

data_dir = Path("data/cwru")
SAMPLE_RATE = 12_000   # samples per second (12 kHz)

# file number -> class name   (0 HP motor load, drive-end sensor)
FILES = {
    97:  "Normal",
    105: "IR_007", 118: "B_007", 130: "OR_007",
    169: "IR_014", 185: "B_014", 197: "OR_014",
    209: "IR_021", 222: "B_021", 234: "OR_021",
}
CLASS_NAMES = list(FILES.values())   # index in this list = class label (0..9)

# Same classes, recorded at 1 HP motor load -> used only in Part 6
FILES_1HP = {
    98:  "Normal",
    106: "IR_007", 119: "B_007", 131: "OR_007",
    170: "IR_014", 186: "B_014", 198: "OR_014",
    210: "IR_021", 223: "B_021", 235: "OR_021",
}

# make results repeatable
random.seed(42)
np.random.seed(42)

In [ ]:
from scipy.signal import decimate

# ⚠️ DATA BUG: the Normal files are recorded at 48 kHz, the fault files at 12 kHz!
# (Check: in 97.mat the shaft frequency 1796 RPM / 60 = 30 Hz only appears at 30 Hz if fs = 48 kHz.)
# If we don't fix this, the model learns "slow-looking signal = Normal" -> a fake shortcut.
NORMAL_48K = {97, 98}


def load_signal(file_number):
    """Load the drive-end (DE) vibration signal from one .mat file, always at 12 kHz."""
    mat = loadmat(data_dir / f"{file_number}.mat")       # a dictionary of arrays
    key = [k for k in mat if k.endswith("_DE_time")][0]  # e.g. "X097_DE_time"
    sig = mat[key].flatten()                             # shape: (N, 1) -> (N,)
    if file_number in NORMAL_48K:
        sig = decimate(sig, 4)                           # 48 kHz -> 12 kHz (low-pass filter + keep every 4th)
    return sig


signals = {name: load_signal(num) for num, name in FILES.items()}

for name, sig in signals.items():
    print(f"{name:7s} samples: {len(sig):7d}   seconds: {len(sig)/SAMPLE_RATE:5.1f}   "
          f"min: {sig.min():6.2f}   max: {sig.max():5.2f}")

We plot only **0.1 seconds** (1200 samples). The full signal is too long to see anything.

In [ ]:
show = ["Normal", "IR_021", "B_021", "OR_021"]
n = int(0.1 * SAMPLE_RATE)                 # 1200 samples = 0.1 s
t = np.arange(n) / SAMPLE_RATE             # time axis in seconds

fig, axes = plt.subplots(len(show), 1, figsize=(12, 8), sharex=True, sharey=True)
for ax, name in zip(axes, show):
    ax.plot(t, signals[name][:n], linewidth=0.7)
    ax.set_title(name, loc="left")
    ax.set_ylabel("accel (g)")
axes[-1].set_xlabel("time (s)")
plt.tight_layout()
plt.show()

**Two simple numbers:**
- **RMS** = "average energy". More damage → more energy.
- **Kurtosis** = how "spiky" the signal is. Smooth noise ≈ 3. Impacts from a crack → much higher.

In [ ]:
from scipy.stats import kurtosis, skew

print(f"{'class':7s}  {'RMS':>6s}  {'kurtosis':>8s}")
for name, sig in signals.items():
    rms = np.sqrt(np.mean(sig ** 2))
    k = kurtosis(sig, fisher=False)         # fisher=False -> normal distribution = 3
    print(f"{name:7s}  {rms:6.3f}  {k:8.2f}")

# Part 2 – Windows and a leak-free split
A model needs many examples, not 10 long recordings. So we **cut each signal into windows** of 1024 samples (≈ 0.085 s ≈ 2.5 shaft turns).

⚠️ **Data leakage trap:** if we cut windows first and then split *randomly*, overlapping windows (almost the same data!) end up in both train and test → fake 100% accuracy.

✅ **Our rule:** split each recording **by time** first: first 60% → train, next 20% → validation, last 20% → test. Then cut windows inside each part.

In [ ]:
WINDOW = 1024
SIGNAL_LEN = 120_000   # use (up to) the first 10 s of every recording
                       # Normal has only ~5 s after the 48k->12k fix -> about half the windows of other classes


def make_windows(sig, stride):
    """Cut a 1D signal into windows of length WINDOW. stride = step between window starts."""
    n = (len(sig) - WINDOW) // stride + 1
    return np.stack([sig[i * stride : i * stride + WINDOW] for i in range(n)])   # shape: (n, WINDOW)


def split_signal(sig):
    """Split by TIME: first 60% train, next 20% validation, last 20% test."""
    sig = sig[:SIGNAL_LEN]
    a, b = int(0.6 * len(sig)), int(0.8 * len(sig))
    return sig[:a], sig[a:b], sig[b:]


def build_split(part_index, stride):
    """part_index: 0 = train, 1 = val, 2 = test. Returns windows X and labels y for all classes."""
    X, y = [], []
    for label, name in enumerate(CLASS_NAMES):
        part = split_signal(signals[name])[part_index]
        w = make_windows(part, stride)
        X.append(w)
        y.append(np.full(len(w), label))     # the same label for every window of this class
    return np.concatenate(X), np.concatenate(y)


# Train windows overlap (stride 256) -> more training examples ("free" augmentation).
# Overlap is OK *inside* one set. It is only a problem *between* sets.
X_train, y_train = build_split(0, stride=256)
X_val,   y_val   = build_split(1, stride=512)
X_test,  y_test  = build_split(2, stride=512)

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")   # (N_train, 1024)
print(f"X_val:   {X_val.shape}    y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape}    y_test:  {y_test.shape}")
print("windows per class (train):", np.bincount(y_train))

In [ ]:
# Picture of the split for one signal
sig = signals["OR_021"][:SIGNAL_LEN]
t = np.arange(len(sig)) / SAMPLE_RATE
a, b = int(0.6 * len(sig)), int(0.8 * len(sig))

plt.figure(figsize=(12, 3))
plt.plot(t[:a], sig[:a], linewidth=0.3, label="train (60%)")
plt.plot(t[a:b], sig[a:b], linewidth=0.3, label="validation (20%)")
plt.plot(t[b:], sig[b:], linewidth=0.3, label="test (20%)")
plt.xlabel("time (s)")
plt.title("OR_021: split by time -> no window is shared between sets")
plt.legend(loc="upper right")
plt.show()

# Part 3 – Hand-made features + Random Forest (classic ML)
A Random Forest can't read 1024 raw numbers well. So **we** describe each window with a few smart numbers ("features"), like a technician who says *"loud, spiky, lots of high frequency"*.

First, let's look at the **frequency spectrum** (FFT). It shows *which* frequencies are in the vibration. Faults add energy at special frequencies.

In [ ]:
freqs = np.fft.rfftfreq(WINDOW, d=1 / SAMPLE_RATE)      # shape: (513,)  0 ... 6000 Hz

plt.figure(figsize=(12, 4))
for name in ["Normal", "IR_021", "B_021", "OR_021"]:
    label = CLASS_NAMES.index(name)
    w = X_train[y_train == label][0]                    # first window of this class, shape: (1024,)
    spectrum = np.abs(np.fft.rfft(w))                   # shape: (513,)
    plt.plot(freqs, spectrum, label=name, linewidth=0.8)
plt.yscale("log")
plt.xlabel("frequency (Hz)")
plt.ylabel("amplitude (log)")
plt.title("Frequency spectrum of one window per class")
plt.legend()
plt.show()

In [ ]:
N_BANDS = 16   # split the spectrum 0-6000 Hz into 16 bands of 375 Hz

TIME_FEATURES = ["mean", "std", "rms", "peak", "peak_to_peak",
                 "crest_factor", "kurtosis", "skewness", "shape_factor", "impulse_factor"]
FEATURE_NAMES = TIME_FEATURES + [f"band_{i*375}-{(i+1)*375}Hz" for i in range(N_BANDS)]


def extract_features(w):
    """One window (1024,) -> one feature vector (26,)."""
    rms = np.sqrt(np.mean(w ** 2))
    peak = np.max(np.abs(w))
    mean_abs = np.mean(np.abs(w))

    time_feats = [
        np.mean(w), np.std(w), rms, peak, np.ptp(w),
        peak / rms,                   # crest factor: how big the peaks are compared to the average
        kurtosis(w, fisher=False),    # spikiness
        skew(w),                      # asymmetry
        rms / mean_abs,               # shape factor
        peak / mean_abs,              # impulse factor
    ]

    spectrum = np.abs(np.fft.rfft(w))                  # shape: (513,)
    bands = np.array_split(spectrum, N_BANDS)          # 16 pieces of ~32 values
    band_energy = [np.log(np.sum(b ** 2) + 1e-12) for b in bands]   # log: energies differ a lot in size

    return np.array(time_feats + band_energy)


F_train = np.array([extract_features(w) for w in X_train])   # shape: (N_train, 26)
F_val   = np.array([extract_features(w) for w in X_val])     # shape: (N_val, 26)
F_test  = np.array([extract_features(w) for w in X_test])    # shape: (N_test, 26)
print(F_train.shape, F_val.shape, F_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(F_train, y_train)

print(f"Validation accuracy: {accuracy_score(y_val, rf.predict(F_val)) * 100:.2f}%")
rf_test_pred = rf.predict(F_test)
rf_test_acc = accuracy_score(y_test, rf_test_pred) * 100
print(f"Test accuracy:       {rf_test_acc:.2f}%\n")
print(classification_report(y_test, rf_test_pred, target_names=CLASS_NAMES, digits=3))

In [ ]:
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASS_NAMES)))
    fig, ax = plt.subplots(figsize=(8, 7))
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap="Blues", colorbar=False)
    plt.xticks(rotation=45)
    plt.title(title)
    plt.show()


plot_confusion(y_test, rf_test_pred, "Random Forest – test set")

# Which features did the forest find most useful?
order = np.argsort(rf.feature_importances_)[::-1][:15]     # top 15, biggest first
plt.figure(figsize=(10, 4))
plt.bar([FEATURE_NAMES[i] for i in order], rf.feature_importances_[order])
plt.xticks(rotation=60, ha="right")
plt.title("Random Forest – top 15 feature importances")
plt.tight_layout()
plt.show()

# Part 4 – 1D CNN on raw windows (PyTorch)
Now the model learns its **own** features from the raw signal. Same idea as the CNN for casting images, but in **1 dimension** (time instead of height × width).

- First layer: a **wide kernel (64 samples)** with stride 8 → it looks at a longer piece of signal, like a "frequency filter", and reduces noise.
- Then small kernels (3) → combine patterns.
- `BatchNorm1d` → keeps the numbers in a nice range → faster, more stable training.

In [ ]:
import copy
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
torch.manual_seed(42)

# AMD GPU (ROCm) fix: MIOpen fails to compile its BatchNorm kernel on this PC
# ("'type_traits' file not found"). This makes PyTorch use its own built-in kernels instead.
torch.backends.cudnn.enabled = False

# Normalize with numbers from the TRAIN set only (the test set must stay "unseen")
SIG_MEAN, SIG_STD = X_train.mean(), X_train.std()


def to_tensor(X):
    X = (X - SIG_MEAN) / SIG_STD
    return torch.tensor(X, dtype=torch.float32).unsqueeze(1)    # (N, 1024) -> (N, 1, 1024)  1 = channel


train_ds = TensorDataset(to_tensor(X_train), torch.tensor(y_train))
val_ds   = TensorDataset(to_tensor(X_val),   torch.tensor(y_val))
test_ds  = TensorDataset(to_tensor(X_test),  torch.tensor(y_test))

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

x, y = next(iter(train_loader))
print(f"one batch: x {x.shape}, y {y.shape}")    # x [64, 1, 1024], y [64]

In [ ]:
class Bearing1DCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # wide first kernel: 1024 -> 128 (conv, stride 8) -> 64 (pool)
            nn.Conv1d(1, 16, kernel_size=64, stride=8, padding=28),
            nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(2),         # [B, 16, 64]

            nn.Conv1d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),         # [B, 32, 32]

            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),         # [B, 64, 16]

            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),         # [B, 64, 8]
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),   # average over time: [B, 64, 8] -> [B, 64, 1]
            nn.Flatten(),              # [B, 64]
            nn.Dropout(0.3),           # randomly switch off 30% -> less overfitting
            nn.Linear(64, num_classes) # [B, 10]
        )

    def forward(self, x):              # x: [B, 1, 1024]
        return self.classifier(self.features(x))


cnn = Bearing1DCNN(num_classes=len(CLASS_NAMES)).to(device)
print(cnn(x.to(device)).shape)         # smoke test -> [64, 10]
print(f"parameters: {sum(p.numel() for p in cnn.parameters()):,}")

In [ ]:
def train_step(model, loader, loss_fn, optimizer):
    model.train()
    total_loss, correct, count = 0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)                  # 1. forward
        loss = loss_fn(logits, y)          # 2. loss
        optimizer.zero_grad()              # 3. zero grad
        loss.backward()                    # 4. backward
        optimizer.step()                   # 5. update
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        count += len(y)
    return total_loss / count, correct / count * 100


def eval_step(model, loader, loss_fn):
    model.eval()
    total_loss, correct, count = 0, 0, 0
    with torch.inference_mode():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            total_loss += loss_fn(logits, y).item() * len(y)
            correct += (logits.argmax(dim=1) == y).sum().item()
            count += len(y)
    return total_loss / count, correct / count * 100

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-3)
EPOCHS = 30

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc, best_state = 0, None

for epoch in range(EPOCHS):
    tr_loss, tr_acc = train_step(cnn, train_loader, loss_fn, optimizer)
    va_loss, va_acc = eval_step(cnn, val_loader, loss_fn)
    for k, v in zip(history, [tr_loss, va_loss, tr_acc, va_acc]):
        history[k].append(v)

    # keep the BEST model (by validation), not just the last one
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_state = copy.deepcopy(cnn.state_dict())

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d} | train loss {tr_loss:.4f} acc {tr_acc:6.2f}% "
              f"| val loss {va_loss:.4f} acc {va_acc:6.2f}%")

cnn.load_state_dict(best_state)
print(f"\nBest validation accuracy: {best_val_acc:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="train"); axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(history["train_acc"], label="train"); axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy (%)"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.show()

In [ ]:
def cnn_predict(model, X):
    """Raw windows (N, 1024) -> predicted labels (N,) as a numpy array."""
    model.eval()
    with torch.inference_mode():
        logits = model(to_tensor(X).to(device))    # [N, 10]
    return logits.argmax(dim=1).cpu().numpy()


cnn_test_pred = cnn_predict(cnn, X_test)
cnn_test_acc = accuracy_score(y_test, cnn_test_pred) * 100
print(f"1D CNN test accuracy: {cnn_test_acc:.2f}%\n")
print(classification_report(y_test, cnn_test_pred, target_names=CLASS_NAMES, digits=3))
plot_confusion(y_test, cnn_test_pred, "1D CNN – test set")

# Part 5 – Autoencoder for anomaly detection
**The real-world problem:** in a factory you have *lots* of "normal" data, but almost no examples of every possible fault. So we can't always train a classifier.

**Idea:** train a model to **copy** normal vibration only.
- An **autoencoder** squeezes the input into a few numbers (the "bottleneck") and then rebuilds it.
- It learns to rebuild *normal* very well.
- When a fault arrives, it has never seen this pattern → it rebuilds it badly → **big reconstruction error = alarm** 🚨

🎵 Analogy: someone who only ever heard one song can sing it perfectly. Play a different song, and their copy sounds wrong.

We feed the autoencoder the **log spectrum** of each window (513 numbers) instead of the raw signal. It is easier to rebuild and less sensitive to small time shifts.

In [ ]:
def to_spectrum(X):
    """Windows (N, 1024) -> log magnitude spectrum (N, 513)."""
    return np.log1p(np.abs(np.fft.rfft(X, axis=1)))


# Normal data: the full recording (~5 s), split by time again.
# Small stride = many overlapping windows -> more training examples from little data.
normal_full = signals["Normal"]
a, b = int(0.6 * len(normal_full)), int(0.8 * len(normal_full))
S_norm_train = to_spectrum(make_windows(normal_full[:a], stride=64))    # train: Normal ONLY
S_norm_val   = to_spectrum(make_windows(normal_full[a:b], stride=256))  # to choose the threshold
S_norm_test  = to_spectrum(make_windows(normal_full[b:], stride=256))   # must NOT raise alarms

# Faults: the test windows of all 9 fault classes (never seen by the autoencoder)
fault_mask = y_test != 0
S_fault_test = to_spectrum(X_test[fault_mask])
y_fault_test = y_test[fault_mask]

# Standardize every frequency bin with Normal-train statistics
SPEC_MEAN = S_norm_train.mean(axis=0)
SPEC_STD = S_norm_train.std(axis=0) + 1e-8

def spec_tensor(S):
    return torch.tensor((S - SPEC_MEAN) / SPEC_STD, dtype=torch.float32)

print(f"normal train: {S_norm_train.shape}  normal val: {S_norm_val.shape}  "
      f"normal test: {S_norm_test.shape}  fault test: {S_fault_test.shape}")

In [ ]:
class SpectrumAutoencoder(nn.Module):
    def __init__(self, n_in=513, bottleneck=8):
        super().__init__()
        self.encoder = nn.Sequential(          # squeeze: 513 -> 128 -> 32 -> 8
            nn.Linear(n_in, 128), nn.ReLU(),
            nn.Linear(128, 32), nn.ReLU(),
            nn.Linear(32, bottleneck),
        )
        self.decoder = nn.Sequential(          # rebuild: 8 -> 32 -> 128 -> 513
            nn.Linear(bottleneck, 32), nn.ReLU(),
            nn.Linear(32, 128), nn.ReLU(),
            nn.Linear(128, n_in),
        )

    def forward(self, x):                      # x: [B, 513]
        return self.decoder(self.encoder(x))   # [B, 513]  (same shape as the input!)


torch.manual_seed(42)
ae = SpectrumAutoencoder().to(device)
ae_loader = DataLoader(TensorDataset(spec_tensor(S_norm_train)), batch_size=64, shuffle=True)
ae_optimizer = torch.optim.Adam(ae.parameters(), lr=1e-3)
mse = nn.MSELoss()

for epoch in range(100):
    ae.train()
    total = 0
    for (xb,) in ae_loader:                    # TensorDataset with ONE tensor -> a tuple of 1
        xb = xb.to(device)
        loss = mse(ae(xb), xb)                 # the target is the input itself!
        ae_optimizer.zero_grad()
        loss.backward()
        ae_optimizer.step()
        total += loss.item() * len(xb)
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d} | reconstruction loss {total / len(ae_loader.dataset):.4f}")

In [ ]:
def reconstruction_error(S):
    """Spectra (N, 513) -> one error number per window (N,)."""
    ae.eval()
    with torch.inference_mode():
        x = spec_tensor(S).to(device)
        err = ((ae(x) - x) ** 2).mean(dim=1)   # mean squared error per window
    return err.cpu().numpy()


err_norm_val = reconstruction_error(S_norm_val)
err_norm_test = reconstruction_error(S_norm_test)
err_fault_test = reconstruction_error(S_fault_test)

# Threshold: 99% of NORMAL validation windows are below it
THRESHOLD = np.percentile(err_norm_val, 99)
print(f"Threshold: {THRESHOLD:.3f}\n")

false_alarm = (err_norm_test > THRESHOLD).mean() * 100
print(f"False alarms on Normal test windows: {false_alarm:.1f}%")
print(f"Detected faults (all fault windows): {(err_fault_test > THRESHOLD).mean() * 100:.1f}%\n")
for label in range(1, len(CLASS_NAMES)):
    m = y_fault_test == label
    print(f"  {CLASS_NAMES[label]:7s} detected: {(err_fault_test[m] > THRESHOLD).mean() * 100:5.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: histogram of errors
bins = np.logspace(np.log10(min(err_norm_test.min(), err_fault_test.min())),
                   np.log10(err_fault_test.max()), 50)
axes[0].hist(err_norm_test, bins=bins, alpha=0.7, label="Normal")
axes[0].hist(err_fault_test, bins=bins, alpha=0.7, label="Faults")
axes[0].axvline(THRESHOLD, color="red", linestyle="--", label="threshold")
axes[0].set_xscale("log")
axes[0].set_xlabel("reconstruction error (log)")
axes[0].set_title("Normal vs faulty windows")
axes[0].legend()

# Right: a "health monitor" like on a factory dashboard
timeline = np.concatenate([err_norm_test, err_fault_test])
axes[1].plot(timeline, ".", markersize=3)
axes[1].axhline(THRESHOLD, color="red", linestyle="--", label="alarm threshold")
axes[1].axvline(len(err_norm_test), color="gray", linestyle=":")
axes[1].text(len(err_norm_test) * 0.1, THRESHOLD * 1.5, "healthy", color="green")
axes[1].text(len(err_norm_test) * 1.5, THRESHOLD * 1.5, "faults (9 types)", color="red")
axes[1].set_yscale("log")
axes[1].set_xlabel("window")
axes[1].set_title("Health monitor: reconstruction error over windows")
axes[1].legend()
plt.tight_layout()
plt.show()

# Part 6 – Robustness test: a different motor load (1 HP)
Until now, train and test came from the **same recordings**. That is easy mode.

In a real factory, the machine load changes. So we test all 3 models on **new recordings at 1 HP load**, which they have never seen. This is the honest test.

In [ ]:
signals_1hp = {name: load_signal(num) for num, name in FILES_1HP.items()}

X_1hp, y_1hp = [], []
for label, name in enumerate(CLASS_NAMES):
    w = make_windows(signals_1hp[name][:SIGNAL_LEN], stride=512)
    X_1hp.append(w)
    y_1hp.append(np.full(len(w), label))
X_1hp, y_1hp = np.concatenate(X_1hp), np.concatenate(y_1hp)
print(f"X_1hp: {X_1hp.shape}")    # (2340, 1024)

rf_1hp_pred = rf.predict(np.array([extract_features(w) for w in X_1hp]))
cnn_1hp_pred = cnn_predict(cnn, X_1hp)


def location(labels):
    """10 classes -> 4 fault LOCATIONS: 'Normal', 'IR', 'B', 'OR'  (e.g. 'B_021' -> 'B')"""
    return np.array([CLASS_NAMES[l].split("_")[0] for l in labels])


# For a technician, "WHERE is the fault?" (which part to replace) matters more than the exact size
print(f"{'model':15s} {'0HP 10-class':>13s} {'1HP 10-class':>13s} {'1HP location':>13s}")
for name, test_pred, pred_1hp in [("Random Forest", rf_test_pred, rf_1hp_pred),
                                  ("1D CNN", cnn_test_pred, cnn_1hp_pred)]:
    acc_0 = accuracy_score(y_test, test_pred) * 100
    acc_1 = accuracy_score(y_1hp, pred_1hp) * 100
    loc_1 = accuracy_score(location(y_1hp), location(pred_1hp)) * 100
    print(f"{name:15s} {acc_0:12.1f}% {acc_1:12.1f}% {loc_1:12.1f}%")

plot_confusion(y_1hp, rf_1hp_pred, "Random Forest – NEW recordings at 1 HP load")
plot_confusion(y_1hp, cnn_1hp_pred, "1D CNN – NEW recordings at 1 HP load")

In [ ]:
# Autoencoder at 1 HP, with the OLD threshold (set from 0 HP healthy data only)
err_1hp = reconstruction_error(to_spectrum(X_1hp))
print(f"Old threshold {THRESHOLD:.2f}:")
print(f"  detected faults:       {(err_1hp[y_1hp != 0] > THRESHOLD).mean() * 100:5.1f}%")
print(f"  false alarms (Normal): {(err_1hp[y_1hp == 0] > THRESHOLD).mean() * 100:5.1f}%   <- way too many!")
print(f"\nMedian error  Normal 0 HP: {np.median(err_norm_test):.2f}   Normal 1 HP: {np.median(err_1hp[y_1hp == 0]):.2f}"
      f"   faults 1 HP: {np.median(err_1hp[y_1hp != 0]):.2f}")

### 6b – Re-calibrate the alarm threshold for the new load
The old threshold came from **5 seconds of healthy data at one load**. That's too tight: a healthy machine at a new load already looks "different".

But look at the medians above: healthy at 1 HP (≈5) is still **far below** any fault (≈16–95). So the model is fine; only the *threshold* is wrong.

**Real-world solution:** when you install the system, you record a short **healthy baseline at every operating condition** and set the threshold from it. We do that here: the first half of the 1 HP Normal recording = baseline, the second half = "new" healthy data to test false alarms.

⚠️ We do **not** retrain the autoencoder, and we never look at fault data to choose the threshold. Otherwise we'd be cheating.

In [ ]:
normal_1hp = signals_1hp["Normal"][:SIGNAL_LEN]
half = len(normal_1hp) // 2

err_baseline_1hp = reconstruction_error(to_spectrum(make_windows(normal_1hp[:half], stride=256)))
err_new_normal_1hp = reconstruction_error(to_spectrum(make_windows(normal_1hp[half:], stride=512)))
err_faults_1hp = err_1hp[y_1hp != 0]

# New threshold: the highest of (old threshold, 99th percentile of the new healthy baseline)
THRESHOLD_1HP = max(THRESHOLD, np.percentile(err_baseline_1hp, 99))

print(f"New threshold for 1 HP: {THRESHOLD_1HP:.2f}")
print(f"  detected faults:       {(err_faults_1hp > THRESHOLD_1HP).mean() * 100:5.1f}%")
print(f"  false alarms (Normal): {(err_new_normal_1hp > THRESHOLD_1HP).mean() * 100:5.1f}%")
print(f"  lowest fault error:    {err_faults_1hp.min():.2f}   highest healthy error: {err_new_normal_1hp.max():.2f}")

# Part 7 – Save the models

In [ ]:
import joblib

model_dir = Path("models")
model_dir.mkdir(exist_ok=True)

torch.save(cnn.state_dict(), model_dir / "bearing_cnn.pth")
torch.save(ae.state_dict(), model_dir / "bearing_autoencoder.pth")
joblib.dump(rf, model_dir / "bearing_random_forest.joblib")

# The autoencoder also needs its normalization numbers and threshold to work later
np.savez(model_dir / "bearing_ae_config.npz", spec_mean=SPEC_MEAN, spec_std=SPEC_STD, threshold=THRESHOLD)
print("saved:", [p.name for p in model_dir.glob("bearing_*")])

# Summary and interview story 🎤

**What I built**
1. Cut vibration signals into windows with a **leak-free, time-based split** (many online tutorials get this wrong).
2. **Classic ML:** 26 physics-inspired features (RMS, kurtosis, crest factor, FFT band energies) + Random Forest.
3. **Deep learning:** a 1D CNN that learns features directly from the raw signal.
4. **Anomaly detection:** an autoencoder trained on *healthy data only*. It raises an alarm when it can't rebuild the vibration. This works even for faults it has never seen.
5. **Robustness test** on a different motor load, to check the models generalize.

**Two real problems I found and fixed (the best interview material!)**
- 🐛 **A data bug:** the Normal recordings are sampled at 48 kHz, the fault recordings at 12 kHz. I found it with physics: the shaft frequency (RPM / 60 = 30 Hz) only appeared in the right place at 48 kHz. Without the fix, the models learned a shortcut ("slow-looking signal = healthy").
- 🎚️ **The alarm threshold:** an autoencoder trained at one motor load raised false alarms on *every* healthy window at a new load. The model still separated healthy from faulty very clearly. The fix was to **calibrate the threshold with a short healthy baseline per operating condition**, which is how you would commission a real system.

**Honest limits (say these in an interview, it shows maturity)**
- CWRU faults are *artificially made* (drilled by EDM) on a lab test rig. Real faults grow slowly.
- This data shows *fault vs. no fault*, not *how much time until failure*. For that (Remaining Useful Life), the next step is a **run-to-failure** dataset like IMS or PRONOSTIA/FEMTO.
- The test windows come from the same recordings as training. That is why Part 6 matters.

**The story:**
> "I used to fix machines after they broke. In this project I detect bearing faults from vibration *before* failure: a classic ML baseline, a 1D CNN, and an autoencoder that only needs healthy data, which is the realistic case in a factory."